# CCSKDE — Full Experiment Pipeline

**Context-Conditioned Sequential Keypoint Density Estimation for Traffic Hazard Detection**

This notebook runs all experiments end-to-end:
1. **Phase 1**: Baseline SeeKer reproduction (10 epochs)
2. **Phase 2**: YOLOv11 context extraction (test + train splits)
3. **Phase 3**: CCSKDE training (10 epochs)
4. **Phase 4**: CCSKDE shuffled-context ablation (10 epochs)
5. **Phase 5**: Results summary

### Prerequisites
Upload these 3 zips to `My Drive/CCSKDE/`:
- `ccskde_code_and_poses.zip`
- `ccskde_test_frames.zip`
- `ccskde_train_videos.zip`

### Runtime
Select **GPU → T4** (or better) in Runtime → Change runtime type.

---
## Phase 0 — Setup

In [ ]:
# ── 0.1  Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 0.2  Check GPU ──
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu}  |  VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('No GPU detected — change runtime type to GPU!')

In [ ]:
# ── 0.3  Install extra dependencies ──
!pip install -q ultralytics einops

In [ ]:
# ── 0.4  Define paths ──
import os, shutil

DRIVE_DIR   = '/content/drive/MyDrive/CCSKDE'
PROJECT     = '/content/CCSKDE'
DATA_DIR    = f'{PROJECT}/data'
RESULTS_DIR = f'{DRIVE_DIR}/results'

os.makedirs(PROJECT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(f'{PROJECT}/logs', exist_ok=True)

print(f'Drive dir:   {DRIVE_DIR}')
print(f'Project dir: {PROJECT}')
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
# ── 0.5  Extract code + poses + GT + YOLO weights ──
!unzip -qo {DRIVE_DIR}/ccskde_code_and_poses.zip -d {PROJECT}
print('Code + poses extracted.')
!ls {PROJECT}/seeker/*.py | head -5
!ls {PROJECT}/ccskde/
!echo '---'
!echo "Train pose files: $(ls {PROJECT}/data/ShanghaiTech/pose/train/*tracked_person.json | wc -l)"
!echo "Test  pose files: $(ls {PROJECT}/data/ShanghaiTech/pose/test/*tracked_person.json | wc -l)"
!echo "GT dir: $(ls {PROJECT}/data/ShanghaiTech/gt/)"

---
## Phase 1 — Baseline SeeKer Reproduction (10 epochs)

This needs only pose data (already extracted). Uses the paper's default hyperparameters:
- batch_size=1024, seg_len=24, lr=5e-4, Adam, 10 epochs
- Target: AUROC ≥ 0.84 (paper reports 0.855)

In [ ]:
# ── 1.1  Run baseline training ──
os.makedirs(f'{PROJECT}/exp_dir', exist_ok=True)

%cd {PROJECT}/seeker
!python -u seeker.py \
    --dataset ShanghaiTech \
    --data_dir {DATA_DIR} \
    --exp_dir {PROJECT}/exp_dir \
    --device cuda \
    --num_workers 2 \
    --epochs 10 \
    --batch_size 1024 \
    --seg_len 24 \
    --seed 42 \
    2>&1 | tee {PROJECT}/logs/baseline_run.log

In [ ]:
# ── 1.2  Extract baseline results & save to Drive ──
import re

with open(f'{PROJECT}/logs/baseline_run.log') as f:
    log = f.read()

# Parse all AUC values
aucs = re.findall(r'AUC on val:\s*([\d.]+)', log)
aucs = [float(a) for a in aucs]

# Parse final loss per epoch
losses = re.findall(r'Loss:\s*([\d.e+-]+)', log)
last_loss = float(losses[-1]) if losses else None

print(f'Baseline AUCs per epoch: {aucs}')
print(f'Best AUC: {max(aucs) if aucs else "N/A"}')
print(f'Final loss: {last_loss}')

# Save checkpoint + log to Drive
!cp {PROJECT}/logs/baseline_run.log {RESULTS_DIR}/

# Find and copy the best checkpoint
import glob
ckpts = glob.glob(f'{PROJECT}/exp_dir/ShanghaiTech/*/checkpoint_best.pth')
if ckpts:
    os.makedirs(f'{RESULTS_DIR}/checkpoints', exist_ok=True)
    shutil.copy(ckpts[-1], f'{RESULTS_DIR}/checkpoints/baseline_best.pth')
    print(f'Saved baseline checkpoint to Drive: {ckpts[-1]}')
else:
    print('WARNING: No baseline checkpoint found')

---
## Phase 2 — YOLOv11 Context Extraction

Runs YOLOv11-nano over all ShanghaiTech clips to build the $C_t$ cache.
- Test split: pre-extracted jpg frames (~4.3 GB)
- Train split: .avi video files (~2.3 GB)

In [ ]:
# ── 2.1  Extract test frames from Drive ──
!unzip -qo {DRIVE_DIR}/ccskde_test_frames.zip -d {PROJECT}
!echo "Test clip dirs: $(ls {PROJECT}/data/ShanghaiTech/shanghaitech/testing/frames/ | wc -l)"

In [ ]:
# ── 2.2  Extract training videos from Drive ──
!unzip -qo {DRIVE_DIR}/ccskde_train_videos.zip -d {PROJECT}
!echo "Train videos: $(ls {PROJECT}/data/ShanghaiTech/shanghaitech/training/videos/*.avi | wc -l)"

In [ ]:
# ── 2.3  Run YOLO extraction — test split ──
%cd {PROJECT}
!PYTHONPATH=. python -u scripts/extract_yolo_detections.py \
    --data-root data/ShanghaiTech \
    --split test \
    --weights yolo11n.pt \
    2>&1 | tee {PROJECT}/logs/yolo_test.log

In [ ]:
# ── 2.4  Run YOLO extraction — train split ──
%cd {PROJECT}
!PYTHONPATH=. python -u scripts/extract_yolo_detections.py \
    --data-root data/ShanghaiTech \
    --split train \
    --weights yolo11n.pt \
    2>&1 | tee {PROJECT}/logs/yolo_train.log

In [ ]:
# ── 2.5  Verify context caches ──
import os
test_npy = len([f for f in os.listdir(f'{PROJECT}/data/ShanghaiTech/context/test') if f.endswith('.npy')])
train_npy = len([f for f in os.listdir(f'{PROJECT}/data/ShanghaiTech/context/train') if f.endswith('.npy')])
print(f'Context cache — test: {test_npy} clips, train: {train_npy} clips, total: {test_npy + train_npy}')
assert test_npy > 100, f'Expected ~107 test caches, got {test_npy}'
assert train_npy > 300, f'Expected ~330 train caches, got {train_npy}'
print('Context cache OK!')

# Save YOLO logs to Drive
!cp {PROJECT}/logs/yolo_test.log {PROJECT}/logs/yolo_train.log {RESULTS_DIR}/

---
## Phase 3 — CCSKDE Training (10 epochs)

Same hyperparameters as baseline, plus the $C_t$ context vector.
If AUROC improves over baseline, the context conditioning is working.

In [ ]:
# ── 3.1  Run CCSKDE training ──
os.makedirs(f'{PROJECT}/exp_dir_ccskde', exist_ok=True)

%cd {PROJECT}
!PYTHONPATH=. python -u ccskde/seeker_ctx.py \
    --dataset ShanghaiTech \
    --data_dir {DATA_DIR} \
    --exp_dir {PROJECT}/exp_dir_ccskde \
    --device cuda \
    --num_workers 2 \
    --epochs 10 \
    --batch_size 1024 \
    --seg_len 24 \
    --seed 42 \
    --context_cache_dir {PROJECT}/data/ShanghaiTech/context \
    2>&1 | tee {PROJECT}/logs/ccskde_run.log

In [ ]:
# ── 3.2  Extract CCSKDE results & save to Drive ──
with open(f'{PROJECT}/logs/ccskde_run.log') as f:
    log = f.read()

aucs_ccskde = re.findall(r'AUC on val:\s*([\d.]+)', log)
aucs_ccskde = [float(a) for a in aucs_ccskde]
print(f'CCSKDE AUCs per epoch: {aucs_ccskde}')
print(f'Best AUC: {max(aucs_ccskde) if aucs_ccskde else "N/A"}')

!cp {PROJECT}/logs/ccskde_run.log {RESULTS_DIR}/

ckpts = glob.glob(f'{PROJECT}/exp_dir_ccskde/ShanghaiTech/*/checkpoint_best.pth')
if ckpts:
    shutil.copy(ckpts[-1], f'{RESULTS_DIR}/checkpoints/ccskde_best.pth')
    print(f'Saved CCSKDE checkpoint to Drive')
else:
    print('WARNING: No CCSKDE checkpoint found')

---
## Phase 4 — Shuffled-Context Ablation (10 epochs)

**Negative control**: permute $C_t$ across segments at training time.
If this gets the same AUC as real context, the spatial-temporal alignment
signal is not contributing — only the extra parameters are.

In [ ]:
# ── 4.1  Run CCSKDE shuffled-context ablation ──
os.makedirs(f'{PROJECT}/exp_dir_shuffled', exist_ok=True)

%cd {PROJECT}
!PYTHONPATH=. python -u ccskde/seeker_ctx.py \
    --dataset ShanghaiTech \
    --data_dir {DATA_DIR} \
    --exp_dir {PROJECT}/exp_dir_shuffled \
    --device cuda \
    --num_workers 2 \
    --epochs 10 \
    --batch_size 1024 \
    --seg_len 24 \
    --seed 42 \
    --context_cache_dir {PROJECT}/data/ShanghaiTech/context \
    --shuffled_context \
    2>&1 | tee {PROJECT}/logs/shuffled_run.log

In [ ]:
# ── 4.2  Extract shuffled results & save to Drive ──
with open(f'{PROJECT}/logs/shuffled_run.log') as f:
    log = f.read()

aucs_shuffled = re.findall(r'AUC on val:\s*([\d.]+)', log)
aucs_shuffled = [float(a) for a in aucs_shuffled]
print(f'Shuffled AUCs per epoch: {aucs_shuffled}')
print(f'Best AUC: {max(aucs_shuffled) if aucs_shuffled else "N/A"}')

!cp {PROJECT}/logs/shuffled_run.log {RESULTS_DIR}/

ckpts = glob.glob(f'{PROJECT}/exp_dir_shuffled/ShanghaiTech/*/checkpoint_best.pth')
if ckpts:
    shutil.copy(ckpts[-1], f'{RESULTS_DIR}/checkpoints/shuffled_best.pth')
    print(f'Saved shuffled checkpoint to Drive')

---
## Phase 5 — Results Summary

In [ ]:
# ── 5.1  Summary table ──
print('=' * 65)
print('  CCSKDE Experiment Results — ShanghaiTech Campus Dataset')
print('=' * 65)
print()

results = {}

if aucs:
    results['Baseline (SeeKer)'] = aucs
    print(f'  Baseline (SeeKer)          Best AUROC: {max(aucs):.4f}')
    print(f'    Per-epoch: {[f"{a:.4f}" for a in aucs]}')
else:
    print('  Baseline: NO RESULTS')

print()

if aucs_ccskde:
    results['CCSKDE'] = aucs_ccskde
    print(f'  CCSKDE (ours)              Best AUROC: {max(aucs_ccskde):.4f}')
    print(f'    Per-epoch: {[f"{a:.4f}" for a in aucs_ccskde]}')
else:
    print('  CCSKDE: NO RESULTS')

print()

if aucs_shuffled:
    results['CCSKDE (shuffled)'] = aucs_shuffled
    print(f'  CCSKDE (shuffled context)  Best AUROC: {max(aucs_shuffled):.4f}')
    print(f'    Per-epoch: {[f"{a:.4f}" for a in aucs_shuffled]}')
else:
    print('  Shuffled: NO RESULTS')

print()
print('-' * 65)

if aucs and aucs_ccskde:
    delta = max(aucs_ccskde) - max(aucs)
    print(f'  Delta (CCSKDE - Baseline): {delta:+.4f}')
if aucs_ccskde and aucs_shuffled:
    delta2 = max(aucs_ccskde) - max(aucs_shuffled)
    print(f'  Delta (CCSKDE - Shuffled):  {delta2:+.4f}')

print('=' * 65)

In [ ]:
# ── 5.2  Save results JSON to Drive ──
import json

results_json = {
    'baseline_aucs': aucs if aucs else [],
    'baseline_best': max(aucs) if aucs else None,
    'ccskde_aucs': aucs_ccskde if aucs_ccskde else [],
    'ccskde_best': max(aucs_ccskde) if aucs_ccskde else None,
    'shuffled_aucs': aucs_shuffled if aucs_shuffled else [],
    'shuffled_best': max(aucs_shuffled) if aucs_shuffled else None,
    'context_cache_test': test_npy,
    'context_cache_train': train_npy,
    'gpu': gpu,
    'batch_size': 1024,
    'epochs': 10,
    'seg_len': 24,
    'seed': 42,
}

with open(f'{RESULTS_DIR}/experiment_results.json', 'w') as f:
    json.dump(results_json, f, indent=2)

print(f'Results saved to {RESULTS_DIR}/experiment_results.json')
print()
print(json.dumps(results_json, indent=2))

In [ ]:
# ── 5.3  Plot AUC curves ──
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

if aucs:
    ax.plot(range(1, len(aucs)+1), aucs, 'o-', label=f'Baseline (best={max(aucs):.4f})', linewidth=2)
if aucs_ccskde:
    ax.plot(range(1, len(aucs_ccskde)+1), aucs_ccskde, 's-', label=f'CCSKDE (best={max(aucs_ccskde):.4f})', linewidth=2)
if aucs_shuffled:
    ax.plot(range(1, len(aucs_shuffled)+1), aucs_shuffled, '^--', label=f'Shuffled (best={max(aucs_shuffled):.4f})', linewidth=2)

ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('AUROC', fontsize=13)
ax.set_title('ShanghaiTech — Validation AUROC per Epoch', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, 11))
plt.tight_layout()

fig.savefig(f'{RESULTS_DIR}/auroc_curves.png', dpi=150)
plt.show()
print(f'Plot saved to {RESULTS_DIR}/auroc_curves.png')

In [ ]:
# ── 5.4  Copy all logs and context caches to Drive ──
# Copy context caches (small .npy files) so we don't have to re-extract
!cp -r {PROJECT}/data/ShanghaiTech/context {RESULTS_DIR}/context_cache

# Copy all experiment dirs (checkpoints)
for exp_name in ['exp_dir', 'exp_dir_ccskde', 'exp_dir_shuffled']:
    src = f'{PROJECT}/{exp_name}'
    if os.path.exists(src):
        dst = f'{RESULTS_DIR}/{exp_name}'
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'Copied {exp_name} to Drive')

print()
print('All results saved to Drive at:', RESULTS_DIR)
print()
print('Files on Drive:')
for root, dirs, files in os.walk(RESULTS_DIR):
    level = root.replace(RESULTS_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        size_mb = os.path.getsize(os.path.join(root, file)) / 1e6
        print(f'{subindent}{file} ({size_mb:.1f} MB)')

---
## Done!

Download these from `My Drive/CCSKDE/results/`:
- `experiment_results.json` — all AUROC numbers
- `auroc_curves.png` — comparison plot
- `checkpoints/` — best model weights for each experiment
- `*_run.log` — full training logs
- `context_cache/` — YOLO detection caches (reusable)

Share these with Antigravity to finalize the report.